In [51]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_classic.retrievers import EnsembleRetriever
from sentence_transformers import CrossEncoder
from langchain.agents.middleware import wrap_model_call
import subprocess
from pathlib import Path
import sys
from typing import Optional, Literal
from langchain_google_genai import ChatGoogleGenerativeAI
from openai import RateLimitError
from decouple import config
from brain.brain_module import Brain 
import threading

### Loading API Keys

In [52]:
GROQ_API_KEY= config("GROQ_API_KEY")
GROQ_API_KEY_BACKUP= config("GROQ_API_KEY_BACKUP")
GOOGLE_API_KEY= config("GOOGLE_API_KEY")
GOOGLE_API_KEY_BACKUP= config("GOOGLE_API_KEY_BACKUP")

In [53]:
embeddings= HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3128.63it/s]


In [54]:
django_vectorstore= Chroma(
    persist_directory="./django_chroma_db",
    collection_name="django_docs",
    embedding_function=embeddings
)

python_vectorstore= Chroma(
    persist_directory="./django_chroma_db",
    collection_name="python_scripts",
    embedding_function=embeddings
)

In [55]:
django_db= django_vectorstore.get()
python_db=python_vectorstore.get()

django_splits=[
    Document(page_content=text, metadata=meta or {})
    for text, meta in zip(django_db["documents"], django_db["metadatas"])
]

python_splits = [
    Document(page_content=text, metadata=meta or {})
    for text, meta in zip(python_db["documents"], python_db["metadatas"])
]

django_retriever= django_vectorstore.as_retriever(search_kwargs={"k":4})
python_retriever= python_vectorstore.as_retriever(search_kwargs={"k":4})


all_splits= django_splits + python_splits
print(f"Loaded {len(all_splits)} total splits ({len(django_splits)} Django docs + {len(python_splits)} Python codebase).")
bm25_retriever= BM25Retriever.from_documents(all_splits)


Loaded 6361 total splits (5110 Django docs + 1251 Python codebase).


In [56]:
bm25_retriever.k=8

In [57]:
#Creating hybrid retriever
hybrid_retriever= EnsembleRetriever(
    retrievers=[django_retriever, python_retriever, bm25_retriever],
    weights=[0.4,0.3,0.3]#40% django sementic search, 30% python sementic and keyword
    
)

In [58]:
#Reranker block of code
reranker= CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3199.46it/s]


## Generation Tools

In [59]:
DEFAULT_DIR = r"C:\Users\ZENOID\Desktop\Home\home\self_made.projects\AI(created by me) generated projects"

@tool   
def retrieve_django_content(query: str) -> str:
    """
    Search the Django knowledge base for documentation, code examples,
    debugging information, and practical examples.
    Use this tool to gain more context and information needed to answer the users question
    """
    print("Using retriever......")
    docs = hybrid_retriever.invoke(query)

    print("Reranking......")
    pairs = [[query, doc.page_content] for doc in docs]
    scores = reranker.predict(pairs)
    scored_docs = sorted(
        zip(docs, scores),
        key=lambda x: x[1],
        reverse=True
    )
    top_docs = scored_docs[:3]

    return "\n\n".join(doc.page_content for doc, score in top_docs)


def _get_bin_dir(env_path: Path) -> Path:
    """Returns the folder inside a venv where executables live (differs by OS)."""
    return env_path / "Scripts" if sys.platform == "win32" else env_path / "bin"


def _exe(bin_dir: Path, name: str) -> Path:
    """Adds .exe to the executable name only on Windows."""
    return bin_dir / (f"{name}.exe" if sys.platform == "win32" else name)


@tool
def setup_django_project(project_name: str,app_name: Optional[list[str] | str] = None,directory: Optional[str] = None) -> dict:
    """
    Set up a Django project safely.
    This operation is idempotent: existing virtual environments,
    Django projects, and apps are detected and will not be recreated.
    """
    if not project_name or not project_name.strip():
        return {"status": "error","message": "project_name is required"}

    if not directory or directory.strip() in ("", "."):
        directory = DEFAULT_DIR

    target_path = Path(directory) / project_name
    target_path.mkdir(parents=True, exist_ok=True)

    results = {
        "status": "success",
        "project_name": project_name,
        "project_path": str(target_path),
        "virtualenv": None,
        "django": None,
        "project": None,
        "apps": []
    }

    # Creating Virtual environment
    env_path = target_path / "env"
    if env_path.exists():
        results["virtualenv"] = "already_exists"
    else:
        print("Creating virtual environent......")
        result = subprocess.run(
            [sys.executable, "-m", "venv", "env"],cwd=target_path,capture_output=True,text=True
        )
        if result.returncode != 0:
            return {
                "status": "error",
                "step": "create_virtualenv",
                "message": result.stderr[-1000:]
            }
        results["virtualenv"] = "created"

    # Getting executables
    bin_dir = _get_bin_dir(env_path)
    python_exe = _exe(bin_dir, "python")
    pip_exe = _exe(bin_dir, "pip")

    # Checking if Django is already installed
    django_check = subprocess.run(
        [str(python_exe), "-c", "import django"],capture_output=True,text=True
    )
    if django_check.returncode == 0:
        results["django"] = "already_installed"
    else:
        print("Installing Django......")
        install_result = subprocess.run(
            [str(pip_exe), "install", "django"],cwd=target_path,capture_output=True,text=True
        )
        if install_result.returncode != 0:
            return {
                "status": "error",
                "step": "install_django",
                "message": install_result.stderr[-1000:]
            }
        results["django"] = "installed"


    # Creating Django Project
    manage_py = target_path / "manage.py"
    project_package = target_path / project_name
    if manage_py.exists() and project_package.exists():
        results["project"] = "already_exists"
    else:
        print("Creating Django project......")
        django_admin = _exe(bin_dir, "django-admin")
        project_result = subprocess.run(
            [str(django_admin),"startproject",project_name,"."],cwd=target_path,capture_output=True,text=True
        )
        if project_result.returncode != 0:
            return {
                "status": "error",
                "step": "create_project",
                "message": project_result.stderr[-1000:]
            }
        results["project"] = "created"


    # Creating Django Apps
    if app_name:
        print("Creating Django Apps......")
        apps = (app_name if isinstance(app_name, list) else [app_name])
        for app in apps:
            app_path = target_path / app
            if app_path.exists():
                results["apps"].append({
                    "name": app,
                    "status": "already_exists"
                })
                continue
            app_result = subprocess.run(
                [str(python_exe),"manage.py","startapp",app],cwd=target_path,capture_output=True,text=True
            )
            if app_result.returncode != 0:
                results["apps"].append({
                    "name": app,
                    "status": "failed",
                    "error": app_result.stderr[-500:]
                })
                results["status"] = "partial_success"
            else:
                results["apps"].append({
                    "name": app,
                    "status": "created"
                })
    return results

_active_server_process = None


@tool
def manage_server(action: str,project_name: Optional[str] = None,directory: Optional[str] = None) -> dict:
    """
    Starts or stops the active Django development server.
    action must be either "start" or "stop".
    For "start", project_name identifies the Django project.
    For "stop", no project_name is required.
    """
    global _active_server_process
    action = action.strip().lower()
    if action not in ("start", "stop"):
        return {
            "status": "error",
            "action": action,
            "message": "Invalid action. Use 'start' or 'stop'."
        }

    if not directory or directory.strip() in ("", "."):
        directory = DEFAULT_DIR

    results = {
        "action": action,
        "project_name": project_name,
        "project_path": None,
        "status": "success"
    }
    # Stops Django Server
    if action == "stop":
        if _active_server_process and _active_server_process.poll() is None:
            _active_server_process.terminate()
            try:
                _active_server_process.wait(timeout=3)
            except subprocess.TimeoutExpired:
                _active_server_process.kill()
            _active_server_process = None
            results["message"] = (
                "Django development server stopped successfully."
            )
        else:
            results["status"] = "not_running"
            results["message"] = (
                "No active Django server is currently running."
            )
        return results


    if not project_name:
        return {
            "status": "error",
            "action": "start",
            "message": "project_name is required when starting the server."
        }

    target_path = Path(directory) / project_name

    # Fallback in case directory itself contains manage.py
    if not (target_path / "manage.py").exists():

        if (Path(directory) / "manage.py").exists():
            target_path = Path(directory)

        else:
            return {
                "status": "error",
                "action": "start",
                "project_name": project_name,
                "message": (
                    f"Could not find manage.py for "
                    f"project '{project_name}'."
                )
            }

    manage_py = target_path / "manage.py"

    results["project_path"] = str(target_path)
    env_path = target_path / "env"
    python_exe = _exe(_get_bin_dir(env_path),"python")

    if not python_exe.exists():

        return {
            "status": "error",
            "action": "start",
            "project_name": project_name,
            "project_path": str(target_path),
            "message": (
                "Could not find the virtual environment Python executable."
            )
        }

    #Checking if Server is running
    if _active_server_process and _active_server_process.poll() is None:
        return {
            "status": "already_running",
            "action": "start",
            "project_name": project_name,
            "project_path": str(target_path),
            "message": (
                "A Django development server is already running."
            )
        }

    #Start Django Server

    try:
        _active_server_process = subprocess.Popen([str(python_exe),"manage.py","runserver","--noreload"],cwd=target_path,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
        results["status"] = "started"
        results["message"] = f"Django server started successfully for project '{project_name}'."
        return results
    except Exception as error:
        return {
            "status": "error",
            "action": "start",
            "project_name": project_name,
            "project_path": str(target_path),
            "message": str(error)
        }

@tool
def run_django_commands(
    command: Literal["makemigrations", "migrate", "collectstatic", "test", "flush", "check"],
    project_name: Optional[str] = None,
    directory: Optional[str] = None,
    app_label: Optional[str] = None,
    no_input: bool = False,
    verbose: bool = False
) -> dict:
    """
    Run Django management commands on a Django project.
    Supported commands: makemigrations, migrate, collectstatic, test, flush, check.
    Parameters:
    - command: The Django management command to run
    - project_name: Name of the Django project (if not provided, uses directory itself)
    - directory: Directory containing the project (defaults to DEFAULT_DIR)
    - app_label: Specific app to target (for makemigrations and migrate)
    - no_input: Skip prompts, use defaults (for collectstatic and flush)
    - verbose: Increase verbosity of output (for test, migrate, etc.)
    Returns: Command output and status
    """
    print(f"Running Django command: {command}......")
    
    if not directory or directory.strip() in ("", "."):
        directory = DEFAULT_DIR
    
    command = command.strip().lower()
    
    # If project_name is provided, use it; otherwise try current directory
    if project_name:
        target_path = Path(directory) / project_name
    else:
        target_path = Path(directory)
    
    # Fallback: if manage.py doesn't exist at target_path, try directory itself
    if not (target_path / "manage.py").exists():
        if (Path(directory) / "manage.py").exists():
            target_path = Path(directory)
        else:
            return {
                "status": "error",
                "error_code": "MANAGE_PY_NOT_FOUND",
                "project_name": project_name,
                "directory": str(target_path),
                "message": f"Could not find manage.py for project '{project_name}' at {target_path}"
            }
    
    # Get Python executable from venv
    env_path = target_path / "env"
    if not env_path.exists():
        return {
            "status": "error",
            "error_code": "VENV_NOT_FOUND",
            "project_name": project_name,
            "project_path": str(target_path),
            "message": "Virtual environment not found. Cannot run commands."
        }
    
    bin_dir = _get_bin_dir(env_path)
    python_exe = _exe(bin_dir, "python")
    
    if not python_exe.exists():
        return {
            "status": "error",
            "error_code": "PYTHON_EXE_NOT_FOUND",
            "project_name": project_name,
            "project_path": str(target_path),
            "message": "Could not find Python executable in virtual environment"
        }
    
    # Build command with whitelisted, hardcoded flags only
    cmd_list = [str(python_exe), "manage.py", command]
    
    # Add command-specific flags based on parameters
    if command == "makemigrations":
        if app_label:
            cmd_list.extend(["--app", app_label])
        if verbose:
            cmd_list.append("--verbose=2")
    
    elif command == "migrate":
        if app_label:
            cmd_list.append(app_label)
        if verbose:
            cmd_list.append("--verbose=2")
    
    elif command == "collectstatic":
        if no_input:
            cmd_list.append("--noinput")
        if verbose:
            cmd_list.append("--verbose=2")
    
    elif command == "test":
        if app_label:
            cmd_list.append(app_label)
        if verbose:
            cmd_list.append("--verbose=2")
    
    elif command == "flush":
        if no_input:
            cmd_list.append("--noinput")
    
    elif command == "check":
        if verbose:
            cmd_list.append("--verbose=2")
    
    try:
        result = subprocess.run(
            cmd_list,
            cwd=target_path,
            capture_output=True,
            text=True
        )
        
        if result.returncode == 0:
            return {
                "status": "success",
                "command": command,
                "project_name": project_name,
                "project_path": str(target_path),
                "output": result.stdout if result.stdout else "Command executed successfully",
                "message": f"Django command '{command}' completed successfully"
            }
        else:
            return {
                "status": "error",
                "error_code": "COMMAND_FAILED",
                "command": command,
                "project_name": project_name,
                "project_path": str(target_path),
                "output": result.stderr[-1000:] if result.stderr else "Unknown error",
                "message": f"Django command '{command}' failed"
            }
    
    except Exception as error:
        return {
            "status": "error",
            "error_code": "COMMAND_EXECUTION_FAILED",
            "command": command,
            "project_name": project_name,
            "project_path": str(target_path),
            "message": str(error)
        }


## File Editor Tools
Prompt/Task -> Search -> Create/Read -> Edit -> Write

In [60]:

_file_content_cache: dict = {}
 
SEARCH_CALL_BUDGET = 4  # tune this — 4 is a reasonable "you should have
                         # found it by now" threshold for a single request
 
 
@tool
def search_codebase(
    query: str,
    directory: Optional[str] = None,
    search_scope: Literal["files", "folders"] = "files",
) -> dict:
    """
    Search a Django codebase for a query.
    Use search_scope="files" to search Python file contents.
    Use search_scope="folders" to search folder names.
    Returns matching paths and relevant line information
    without returning entire file contents.
    """
    print("Enabling codebase search...")
 
    if not directory or directory.strip() in ("", "."):
        directory = DEFAULT_DIR
    if not query or not query.strip():
        return {"status": "error", "error_code": "QUERY_REQUIRED", "message": "A search query is required."}
 
    query = query.strip()
    path = Path(directory)
 
    if not path.exists():
        return {"status": "error", "error_code": "DIRECTORY_NOT_FOUND", "directory": str(path),
                 "message": f"Directory does not exist: {path}"}
    if not path.is_dir():
        return {"status": "error", "error_code": "INVALID_DIRECTORY", "directory": str(path),
                 "message": f"The provided path is not a directory: {path}"}
 
    # ---- circuit breaker ----
    ctx = agent_short_brain.get_context()
    call_count = ctx.get("search_call_count", 0) + 1
    agent_short_brain.update_context(search_call_count=call_count)
 
    if call_count > SEARCH_CALL_BUDGET:
        return {
            "status": "search_budget_exceeded",
            "query": query,
            "calls_made": call_count,
            "message": (
                f"You have called search_codebase {call_count} times this session "
                "without locking onto a usable file. STOP searching now. Either "
                "ask the user for the exact file path, or state plainly that the "
                "code could not be located. Do not call search_codebase again "
                "this turn."
            ),
        }
 
    if search_scope == "files":
        print("Searching file contents...")
        matches = []
        for file in path.rglob("*.py"):
            if any(excluded in file.parts for excluded in ("env", "venv", ".venv", "site-packages")):
                continue
            try:
                mtime = file.stat().st_mtime
            except OSError:
                continue
 
            cache_key = str(file)
            cached = _file_content_cache.get(cache_key)
            if cached is not None and cached[0] == mtime:
                lines = cached[1]  # cache hit — no disk read, no decode
            else:
                try:
                    lines = file.read_text(encoding="utf-8").splitlines()
                except (UnicodeDecodeError, PermissionError, OSError):
                    continue
                _file_content_cache[cache_key] = (mtime, lines)  # cache miss — store
 
            for line_number, line in enumerate(lines, start=1):
                if query.lower() in line.lower():
                    matches.append({"path": cache_key, "line": line_number, "preview": line.strip()})
 
        if matches:
            # found something — give the model a fresh budget for its next search
            agent_short_brain.update_context(search_call_count=0)
 
        return {
            "status": "success",
            "query": query,
            "directory": str(path),
            "search_scope": "files",
            "matches": matches,
            "total_matches": len(matches),
        }
 
    elif search_scope == "folders":
        # unchanged — folder scans never read file contents, so they were
        # never the latency problem.
        print("Searching folders...")
        matches = []
        for folder in path.rglob("*"):
            if not folder.is_dir():
                continue
            if any(excluded in folder.parts for excluded in ("env", "venv", ".venv", "site-packages")):
                continue
            if query.lower() in folder.name.lower():
                matches.append({"path": str(folder), "name": folder.name})
        return {
            "status": "success",
            "query": query,
            "directory": str(path),
            "search_scope": "folders",
            "matches": matches,
            "total_matches": len(matches),
        }





@tool
def file_operations(file_path: str, 
                    operation: Literal["read", "edit", "append", "write"],
                    old_code: Optional[str] = None,
                    new_code: Optional[str] = None,
                    overwrite: Optional[bool] = False,
                    start_line: Optional[int]= None,
                    end_line: Optional[int]= None,
                    ) -> dict:
    """
    Perform operations on a file.
    Supported operations:
    read:
        Reads a file. Can read the entire file or a specific
        range of lines using start_line and end_line.
    edit:
        Replaces one exact and unique block of old_code with new_code.
    append:
        Adds new_code to the end of an existing file.
    write:
        Creates a new file with new_code.
        Existing files are not overwritten unless overwrite=True.
    """

    print(f"Performing '{operation}' on {file_path}.......")

    path= Path(file_path)

    operation= operation.strip().lower()

    #Read Operation

    if operation == "read":
        if not path.exists():
            return {
                "status": "error",
                "error_code": "FILE_NOT_FOUND",
                "path": str(path),
            }
        
        try:
            lines= path.read_text(encoding="utf-8").splitlines()
            total_lines= len(lines) #Checking for the total number of files available

            #If theres no line provided for us to start reading we will start from the begining
            if start_line is None:
                start_line= 1

            #Read to the end of the file
            if end_line is None:
                end_line=total_lines

            #Checking if start_line is valid
            if start_line < 1:
                return{
                    "status": "error",
                    "error_code": "INVALID_START_LINE",
                    "path": str(path),
                    "message": "start_line must be greater than or equal to 1"
                }

            #Checking if end_line is valid
            if end_line < start_line:
                return{
                    "status": "error",
                    "error_code": "INVALID_LINE_RANGE",
                    "path": str(path),
                    "message": "end_line cannot be lower than the start_line"
                }

            #Checking if start_line exists
            if start_line > total_lines:
                return{
                    "status": "error",
                    "error_code": "START_LINE_OUT_OF_RANGE",
                    "path": str(path),
                    "total_lines": total_lines,
                }

            #Prevents end_line from going beyond the file
            end_line= min(end_line, total_lines)

            #Minusing it by one because we didnt start from 0 when labelling each match
            selected_lines= lines[start_line -1: end_line]

            #Returnong the line number and the line we read back to the LLM
            content= "\n".join(f"{line_number}: {line}" for line_number, line in enumerate(selected_lines, start=start_line))

            return{
                "status": "success",
                "operation": "read",
                "path": str(path),
                "start_line": start_line,
                "end_line": end_line,
                "total_lines": total_lines,
                "content": content
            }

        except (UnicodeDecodeError, PermissionError, OSError) as error:
            return{
                "status": "error",
                "error_code": "FILE_READ_FAILED",
                "path": str(path),
                "message": str(error)
            }


    #Edit Operation
    elif operation == "edit":
        if not path.exists():
            return{
                "status": "error",
                "error_code": "FILE_NOT_FOUND",
                "path": str(path),
            }
        if not old_code:
            return{
                "status": "error",
                "error_code": "OLD_CODE_REQUIRED",
                "path": str(path),
            }

        if new_code is None:
            return{
                "status": "error",
                "error_code": "NEW_CODE_REQUIRED",
                "path": str(path),
            }
        
        try:
            content= path.read_text(encoding="utf-8")

            count= content.count(old_code)

            if count == 0:
                return {
                    "status": "error",
                    "error_code": "OLD_CODE_NOT_UNIQUE",
                    "path": str(path),
                    "matches": count
                }

            updated_content= content.replace(old_code, new_code, 1)

            path.write_text(updated_content, encoding="utf-8")
            return{
                "status": "modified",
                "operation": "edit",
                "path": str(path),
                "changes_summary": "Successfully replaced the specified code block"
            }
        except (UnicodeDecodeError, PermissionError, OSError) as error:
            return{
                "status": "error",
                "error_code": "FILE_EDIT_FAILED",
                "path": str(path),
                "message": str(error)
            }

    #Append Operation
    elif operation == "append":
        if not path.exists():
            return{
                "status": "error",
                "error_code": "FILE_NOT_FOUND",
                "path": str(path),
            }

        if not new_code:
            return{
                "status": "error",
                "error_code": "NEW_CODE_REQUIRED",
                "path": str(path),
            }

        try:
            with open(path, "a", encoding="utf-8") as f:
                f.write("\n\n" + new_code)
            return{
                "status": "modified",
                "operation": "append",
                "path": str(path),
                "changes_summary": "Successfully appended new code"
            }
        except (PermissionError, OSError) as error:
            return{
                "status": "error",
                "error_code": "FILE_APPEND_FAILED",
                "path": str(path),
                "message": str(error)
            }

    #Write Operation
    elif operation == "write":
        if path.exists() and not overwrite:
            return{
                "status": "error",
                "error_code": "FILE_ALREADY_EXISTS",
                "path": str(path),
                "message": "File already exists at this path. Use overwrite=True if you really want to replace it, or use edit_file/append_to_file to modify it instead."
            }

        if new_code is None:
            return{
                "status": "error",
                "error_code": "NEW_CODE_REQUIRED",
                "path": str(path),
            }
        try:
            path.parent.mkdir(
                parents=True,
                exist_ok=True
            )

            path.write_text(
                new_code,
                encoding="utf-8"
            )

            return{
                "status": "created",
                "operation": "write",
                "path": str(path)
            }
        
        except (PermissionError, OSError) as error:
            return{
                "status": "error",
                "error_code": "FILE_WRITE_FAILED",
                "path": str(path),
                "message": str(error)
            }
    return{
        "status": "error",
        "error_code": "INVALID_OPERATION",
        "operation": operation,
        "message": "Operations must be one of: read, edit, append, or write"
    }


### Brain ToolSet

In [61]:
agent_short_brain = Brain()

@tool
def manage_tasks(
    tasks: list[str], action: Literal["create", "delete"]
) -> dict:
    """Manages the agent task queue.
 
    Args:
        tasks: List of task descriptions to register or manage.
        action: Either 'create' to populate tasks or 'delete' to clear the queue.
    """
    action_clean = action.lower().strip()
 
    if action_clean not in ["create", "delete"]:
        return {
            "status": "error",
            "action": action_clean,
            "message": "Invalid action. Must be 'create' or 'delete'.",
        }
 
    if not tasks and action_clean == "create":
        return {
            "status": "error",
            "message": "'tasks' list cannot be empty when action is 'create'.",
        }
 
    if action_clean == "create":
        try:
            print(f"Creating {len(tasks)} tasks...")
            created = agent_short_brain.create_tasks(tasks)  # ONE batched write
            return {
                "status": "success",
                "message": f"Successfully created {len(created)} tasks.",
                "current_tasks": agent_short_brain.get_full_state()["current_conversation"],
            }
        except Exception as e:
            return {"status": "error", "message": f"Failed to create tasks: {str(e)}"}
 
    elif action_clean == "delete":
        try:
            print("Task completed currently deleting....")
            current_conversation = agent_short_brain.get_full_state()["current_conversation"]
            if not current_conversation:
                return {
                    "status": "warning",
                    "message": "Task queue is already empty.",
                    "current_tasks": [],
                }
            agent_short_brain.clear_tasks()
            return {"status": "success", "message": "All tasks cleared successfully from short-term memory."}
        except Exception as e:
            return {"status": "error", "message": f"Failed to delete tasks: {str(e)}"}


In [62]:
#Deleting memory history/cache
# memory.storage.pop("test-8", None)

### Memory

In [63]:
MAX_RESULT_CHARS = 1200
MAX_TOOL_MESSAGE_CHARS = 1400


def compact_text(value: object, max_chars: int = MAX_RESULT_CHARS) -> str:
    text = str(value)
    if len(text) <= max_chars:
        return text
    return "...[older output trimmed]...\n" + text[-max_chars:]


@wrap_model_call
def limit_history(request, call_next):
    messages = request.messages
    human_indices = [i for i, m in enumerate(messages) if isinstance(m, HumanMessage)]

    KEEP_TURNS = 1
    if len(human_indices) > KEEP_TURNS:
        cut_at = human_indices[-KEEP_TURNS]
        messages = messages[cut_at:]

    compacted_messages = []
    for message in messages:
        if getattr(message, "type", "") == "tool" and isinstance(message.content, str):
            message = message.model_copy(
                update={
                    "content": compact_text(
                        message.content, MAX_TOOL_MESSAGE_CHARS
                    )
                }
            )
        compacted_messages.append(message)

    request = request.override(messages=compacted_messages)
    return call_next(request)


#### AI agent variables, Tools and Prompts

In [64]:
BUILDER_PROMPT = r"""
You are a Django engineering assistant with two jobs: reviewing Django code, and setting up/running Django projects using the tools available to you.

═══════════════════════════════════════════
STEP 0 — ROUTE THE REQUEST BEFORE TOUCHING ANY TOOL
═══════════════════════════════════════════
Before calling anything, classify the request into exactly one of these:

A) BUILD / SCAFFOLD something new (a project, an app, "set this up," "create X")
   -> Go straight to ACTION TOOLS. Do NOT call search_codebase first.
      setup_django_project is idempotent (it already checks internally whether the
      venv, project, or each app already exists, and skips recreating anything
      that's already there) — it needs no pre-check from you. Calling
      search_codebase on a project that doesn't exist yet wastes a call and finds
      nothing, since there's nothing on disk to search.

B) MODIFY / FIX / READ code that already exists (bugs, "my views.py," "add a field to this model")
   -> This is the ONLY case where FILE TOOLS apply. If you don't have the exact
      file path already, use search_codebase to find it. If the user already gave
      you the path, skip search_codebase and call file_operations directly.

C) EXPLAIN / DEBUG / ASK ABOUT a Django concept, error, or pattern (no specific file involved)
   -> Call retrieve_django_content. This is a real knowledge base (a hybrid
      retriever plus a reranker) holding actual Django documentation and a
      complete example codebase — it's more reliable than answering from memory,
      so prefer it over guessing whenever the question is conceptual.

D) RUN a management command or start/stop the server on a project that already exists
   -> Go straight to run_django_commands / manage_server. No search needed unless
      the user is vague about which project/directory they mean.

A single request can span more than one category (e.g. "review this code, then add
a new app to it") — handle each part with only the tools that category needs.

If you're unsure which category a request falls into, default to the cheaper,
more direct tool. Never call search_codebase "just in case" — only call it when
you genuinely lack a file path for code you have reason to believe already exists.

═══════════════════════════════════════════
1. KNOWLEDGE TOOL — retrieve_django_content
═══════════════════════════════════════════
Use for Category C requests: Django concepts, patterns, debugging, best practices,
code reviews/explanations, or "why isn't this working." Do NOT use for pure action
requests (Category A/D) — those need tool execution, not documentation.

WORKFLOW:
1. Call retrieve_django_content with the user's question or code snippet.
2. Answer using what the retriever returns. If it's empty or irrelevant, say
   "I don't have this information in my knowledge base" — do not guess.
3. Lead with the answer, then 1-3 sentences of context. Bullet points only if the
   user explicitly asks for a list.
4. Do NOT narrate tool calls ("I searched the knowledge base...") — just do it and
   report the outcome.

═══════════════════════════════════════════
2. FILE TOOLS — only for Category B (code that already exists on disk)
═══════════════════════════════════════════
- search_codebase(query, search_scope="files"): ONLY when the user refers to
  code you have reason to believe already exists, and you don't have an exact
  path. Never use this to check whether a *new* project/app exists —
  setup_django_project already handles that.
- search_codebase(query, search_scope="folders"): for locating folder structure
  in an existing codebase.
- file_operations(file_path, operation="read"): ALWAYS call before editing or
  appending to any file. Never assume content — it may have changed.
- file_operations(file_path, operation="edit", old_code, new_code): old_code must
  match EXACTLY, including whitespace. If not found or non-unique, re-read and
  adjust — do not retry blindly.
- file_operations(file_path, operation="append", new_code): for adding new code
  (functions, classes, utilities) to the end of a file. Do NOT use for
  strictly-structured files (urls.py, settings.py) — use edit instead.
- file_operations(file_path, operation="write", new_code, overwrite=False):
  create a NEW file only if it doesn't exist. Do not use overwrite=True unless
  the user explicitly asked to replace the whole file.

WORKFLOW FOR CODE CHANGES:
1. If you don't have the exact file path, call search_codebase first.
2. Call file_operations with operation="read" to see current content.
3. Decide: new file (write), new code added (append), or change existing (edit)?
4. Make the change. Report in plain words what changed — do not paste raw tool output.
5. If a step fails, re-read and adjust — do not guess or retry blindly.

═══════════════════════════════════════════
3. ACTION TOOLS — Category A and D. Real actions on the user's machine.
═══════════════════════════════════════════
- setup_django_project(project_name, app_name, directory): Bootstraps a complete
  Django setup in one call (virtualenv, Django install, project, apps). This tool
  is idempotent — it already detects and skips anything that already exists, so
  you never need to search or check beforehand. Just call it.
- manage_server(action, project_name, directory): action="start" or "stop".
- run_django_commands(command, project_name, directory, app_label, no_input, verbose):
  makemigrations, migrate, collectstatic, test, flush, check.

STRICT ORDER OF OPERATIONS (for scaffolding requests):
1. setup_django_project (handles venv + Django + project + apps in one step).
2. manage_server to start/stop as requested.
Skip a step only if the user says it already exists — even then, you can still
just call setup_django_project, since it will detect that and skip it safely.

RULES:
- If the user asks to create, set up, scaffold, start, or run a project/app —
  call the tools directly. Do not describe steps in text instead of calling them.
- Infer sensible defaults if vague (e.g. project name "myproject") and state the
  assumption in one short sentence.
- After any tool call, report what actually happened in 1-2 sentences — summarize,
  don't paste raw tool output.
- If a tool fails or the server doesn't start, say so plainly and suggest the
  likely cause — do not pretend it succeeded.
- Never call manage_server(action="start") without confirming the project exists
  at that path.
- Only call manage_server(action="stop") if the user asks to stop/restart, or a
  server may already be running before you start a new one.
- If the user only asks what steps you'd take, describe them in words — do not
  call tools.

═══════════════════════════════════════════
GENERAL
═══════════════════════════════════════════
- Be concise. No filler.
- Every tool call should be justified by the routing decision in STEP 0 — if you
  can't name which category (A/B/C/D) drove a tool call, don't make it.
- If ambiguous between "explain how to do X" and "do X for me," default to doing
  it if action tools fit — ask only if acting could hit the wrong project/directory.
"""

In [65]:
ORCHESTRATOR_TASK_DETAIL_RULES = r"""
### TASK-WRITING RULES (every task must satisfy ALL of these)
Each task description must name, explicitly, whatever of the following is
already knowable from the user's question:
  - the exact project name and/or app name involved
  - the exact file path, if the user referenced a specific file or one was
    established earlier in this session
  - which family of action it needs: BUILD (scaffold something new),
    MODIFY (change existing code), EXPLAIN (answer a concept question), or
    RUN (a management command / server action)
 
BAD:  "Set up the project."
GOOD: "Run setup_django_project for project 'inventory_app' with app
       'stock', using the default directory."
 
BAD:  "Fix the bug in the view."
GOOD: "In inventory_app/stock/views.py, fix the IndexError in the
       StockListView.get_queryset method (already located in a prior
       task — do not re-search for the file)."
 
If a task requires the builder to search for something the user did not
name and no earlier task has already located, say so explicitly in the
task text ("locate the file handling X before editing it") rather than
leaving it implicit — an implicit expectation is what causes repeated,
unproductive search_codebase calls.
"""

In [66]:
#Constructing Agent and memory
AGENTIC_MODELS=[
    ChatGroq(
        model="openai/gpt-oss-120b",
        api_key=GROQ_API_KEY,
        temperature=0,
        max_tokens=2000,
    ),
    ChatGroq(
        model="openai/gpt-oss-120b",
        api_key=GROQ_API_KEY_BACKUP,
        temperature=0,
        max_tokens=2000,
    ),
]

ORCHESTRATOR_MODELS=[
    ChatGoogleGenerativeAI(
        model="gemini-3.6-flash",
        api_key=GOOGLE_API_KEY,
        temperature=0,
        max_output_tokens=2000,
    ),
    ChatGoogleGenerativeAI(
        model="gemini-3.6-flash",
        api_key=GOOGLE_API_KEY_BACKUP,
        temperature=0,
        max_output_tokens=2000,
    ),
]
memory=MemorySaver()

### Tools, Agent and Brain

### Brain Manager

In [67]:
ORCHESTRATOR_TOOLS = [
    manage_tasks,
]


def initialize_orchestrator(model):
    return create_agent(
        model=model,
        tools=ORCHESTRATOR_TOOLS,
        system_prompt=ORCHESTRATOR_PROMPT,
    )

primary_orchestrator = initialize_orchestrator(ORCHESTRATOR_MODELS[0])
backup_orchestrator = initialize_orchestrator(ORCHESTRATOR_MODELS[1])


def invoke_orchestrator_with_fallback(payload, invoke_config):
    try:
        return primary_orchestrator.invoke(payload, config=invoke_config)
    except RateLimitError:
        print("Primary orchestrator rate limit reached; trying the backup model...")
        return backup_orchestrator.invoke(payload, config=invoke_config)


### Builder agent

In [68]:
BUILDER_CONTEXT_PROMPT = r"""

### SHARED TASK SESSION
You are working inside one continuous build session. The task message may be the next task from the Orchestrator, not a new project.
- Reuse the same project directory, files, and decisions from earlier turns in this thread.
- Read the shared session context included in each task before searching for folders again.
- Do not rediscover the project when the context already identifies it; inspect only what the current task needs.
- Complete only the current task, then report what changed, the relevant project path, and any blocker for the next task.
"""

BUILDER_TOOLS = [
    retrieve_django_content,
    setup_django_project,
    manage_server,
    file_operations,
    search_codebase,
    run_django_commands,
    manage_tasks,
]


def initialize_builder(model):
    return create_agent(
        model=model,
        tools=BUILDER_TOOLS,
        system_prompt=BUILDER_PROMPT + BUILDER_CONTEXT_PROMPT,
        middleware=[limit_history],
        checkpointer=memory,
    )

primary_builder = initialize_builder(AGENTIC_MODELS[0])
backup_builder = initialize_builder(AGENTIC_MODELS[1])

builder_thread_id = "builder-session-2"
builder_config = {"configurable": {"thread_id": builder_thread_id}}


def invoke_builder_with_fallback(payload, invoke_config):
    try:
        return primary_builder.invoke(payload, config=invoke_config)
    except RateLimitError:
        print("Primary builder rate limit reached; trying the backup model...")
        return backup_builder.invoke(payload, config=invoke_config)


In [69]:
def run_request(question: str):
    """Plan one request, then execute its queued tasks on one builder thread."""
    orchestrator_config = {
        "configurable": {"thread_id": "orchestrator-session-1"}
    }

    agent_short_brain.update_context(
        project_directory=agent_short_brain.get_context().get(
            "project_directory", DEFAULT_DIR
        ),
        last_request=compact_text(question),
    )
    invoke_orchestrator_with_fallback(
        {"messages": [HumanMessage(content=question)]},
        orchestrator_config,
    )

    completed = []
    while True:
        task = agent_short_brain.get_next_task()
        if task is None:
            break

        saved_context = agent_short_brain.get_context()
        context = {
            "project_directory": saved_context.get("project_directory", DEFAULT_DIR),
            "last_task": compact_text(saved_context.get("last_task", "")),
            "last_result": compact_text(saved_context.get("last_result", "")),
        }
        handoff = (
            "Continue the same build session. Do not start over or rediscover the "
            "project.\n\n"
            f"Shared session context: {context}\n\n"
            f"Current task: {compact_text(task['description'])}\n\n"
            "Use the existing project path and prior work whenever possible. "
            "When this task is complete, summarize the change and the project path."
        )
        result = invoke_builder_with_fallback(
            {"messages": [HumanMessage(content=handoff)]},
            builder_config,
        )
        summary = compact_text(result["messages"][-1].content)
        agent_short_brain.update_context(
            last_task=compact_text(task["description"]),
            last_result=summary,
        )
        agent_short_brain.mark_task_completed()
        completed.append(summary)

    return completed


### Users Input/Interface alongside model switching

In [72]:
while True:
    try:
        question = input("Any Question about django: ").strip()

        if question.lower() in ["exit", "quit", "q"]:
            print("See you soon...")
            break
        if not question:
            continue

        results = run_request(question)
        if results:
            print(f"\n{results[-1]}")
        else:
            print("No task was created for this request.")
    except RateLimitError:
        print("All models ratelimit exhausted, please try again later")
        break
    except Exception as error:
        print(error)
        break


c:\Users\ZENOID\Desktop\Home\home\self_made.projects\Standard Projects\code_debugger\env\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Creating 22 tasks...


c:\Users\ZENOID\Desktop\Home\home\self_made.projects\Standard Projects\code_debugger\env\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Creating 12 tasks...


c:\Users\ZENOID\Desktop\Home\home\self_made.projects\Standard Projects\code_debugger\env\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


No task was created for this request.


c:\Users\ZENOID\Desktop\Home\home\self_made.projects\Standard Projects\code_debugger\env\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Creating 1 tasks...


c:\Users\ZENOID\Desktop\Home\home\self_made.projects\Standard Projects\code_debugger\env\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


No task was created for this request.
See you soon...
